# 06 — So sánh dev và chọn cấu hình SVM
  Giống KNN: chọn bằng Macro-F1 dev cao nhất, nếu bằng nhau thì MAE thấp nhất.
  Giữ năm biểu diễn và bảy SVC mỗi biểu diễn, tổng 35 cấu hình.

In [ ]:
from pathlib import Path
import os, sys
# Bỏ comment nếu Kaggle gắn nhiều bộ code hoặc dữ liệu.
# os.environ['CS231_PROJECT_ROOT'] = '/kaggle/input/TEN-DATASET-CODE'
# os.environ['CS231_DATA_ROOT'] = '/kaggle/input/TEN-DATASET-ANH/DATA'
# os.environ['CS231_SVM_PREVIOUS_RESULTS'] = '/kaggle/input/.../CS231_SVM/results_224'
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cs231-matplotlib')
roots = [Path.cwd(), *Path.cwd().parents]
roots += [p for pattern in ('*', 'datasets/*/*') for p in Path('/kaggle/input').glob(pattern)]
if os.environ.get('CS231_PROJECT_ROOT'):
    roots = [Path(os.environ['CS231_PROJECT_ROOT'])]
projects = sorted({p.resolve() for p in roots if (p/'src/preprocessing.py').is_file()})
if len(projects) != 1:
    raise FileNotFoundError('Đặt CS231_PROJECT_ROOT trỏ tới thư mục chứa src.')
PROJECT_ROOT = projects[0]
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runtime import setup, restore_previous_output
DATA_ROOT, RESULTS_ROOT = setup('SVM', PROJECT_ROOT)

In [ ]:
restore_previous_output(RESULTS_ROOT, 'SVM', 'hsv12_hog_pca/svm_grid_dev.csv')

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.SVM.pipeline import (
    CLASSES, FEATURE_FOLDERS, manifest_for_split, extract_matrix, l2,
    save_features, load_features, same_manifest, run_dev_grid, save_dev_result,
    save_json, hsv12, hog_raw,
)
from src.preprocessing import read_rgb224

## 1. Tổng hợp đủ 35 cấu hình

In [ ]:
from src.SVM.pipeline import FEATURE_ORDER, candidates
SELECTION = RESULTS_ROOT / 'selection'
SELECTION.mkdir(parents=True, exist_ok=True)
grids = []
for order, feature_id in enumerate(FEATURE_ORDER):
    grid = pd.read_csv(RESULTS_ROOT/FEATURE_FOLDERS[feature_id]/'svm_grid_dev.csv', float_precision='round_trip')
    grid['feature_id'] = feature_id
    grid['feature_order'] = order
    grids.append(grid)
all_results = pd.concat(grids, ignore_index=True).sort_values(
    ['macro_f1', 'mae_vnd', 'feature_order', 'order'], ascending=[False, True, True, True])
all_results.to_csv(SELECTION/'all_35_configurations_dev.csv', index=False)
display(all_results.head(10))

## 2. So sánh biểu diễn và giữ baseline linear C=1

In [ ]:
columns = ['feature_id', 'kernel', 'C', 'gamma', 'accuracy', 'macro_f1', 'mae_vnd']
best_by_feature = all_results.drop_duplicates('feature_id', keep='first')
reference = all_results.loc[(all_results.kernel == 'linear') & (all_results.C == 1.)]
best_by_feature[columns].to_csv(SELECTION/'best_by_feature_dev.csv', index=False)
reference[columns].to_csv(SELECTION/'reference_dev.csv', index=False)
display(best_by_feature[columns])
display(reference[columns])

## 3. Lưu cấu hình đã chọn
  Bước 07 chỉ dùng cấu hình này, fit trên train rồi dự đoán test. Không gộp train và dev.

In [ ]:
best = all_results.iloc[0]
feature_id = str(best.feature_id)
X_train, _, _ = load_features(RESULTS_ROOT/FEATURE_FOLDERS[feature_id], 'train')
gamma = 'scale' if best.gamma == 'scale' else float(best.gamma)
lock = {
    'feature_id': feature_id, 'feature_folder': FEATURE_FOLDERS[feature_id],
    'feature_dim': int(X_train.shape[1]), 'classes_vnd': CLASSES,
    'svm': {'kernel': str(best.kernel), 'C': float(best.C), 'gamma': gamma,
            'cache_size': 512, 'decision_function_shape': 'ovr', 'break_ties': True},
    'dev_metrics': {name: float(best[name]) for name in ('accuracy', 'macro_f1', 'mae_vnd')},
}
save_json(SELECTION/'locked_svm_config.json', lock)
print('Cấu hình chọn:', lock)